# 📊 Volve Field Production Analysis & Portfolio Showcase

## Executive Summary & Key Findings

### 1. Well Bore Output & Performance Rankings
* **Top Producer:** Well bore **`NO 15/9-F-12 H`** is the leading contributor in the field, generating over **4.57 million barrels (bbl)** of total oil.
* **Primary Volume Drivers:** The top two well bores (`NO 15/9-F-12 H` and `NO 15/9-F-14 H`) account for the vast majority of overall field output.
* **Secondary Wells:** Minor producers such as `NO 15/9-F-1 C`, `NO 15/9-F-15 D`, and `NO 15/9-F-5 AH` represent a significantly lower percentage of total output.

### 2. SQL Analytics Methodology
* **Aggregations (`GROUP BY`, `SUM`):** Applied to isolate, summarize, and compare total lifetime output across distinct well bores.
* **Window Functions (`RANK() OVER`, `SUM() OVER`):** Utilized for time-series cumulative production tracking and dynamic well ranking without losing record-level detail.

In [2]:

!pip install kagglehub
import kagglehub

# Download latest version of the Volve production dataset
path = kagglehub.dataset_download("lamyalbert/volve-production-data")

print("Path to dataset files:", path)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [kagglehub]

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


100%|██████████| 1.93M/1.93M [00:00<00:00, 69.7MB/s]

Extracting files...
Path to dataset files: /home/repl/.cache/kagglehub/datasets/lamyalbert/volve-production-data/versions/1


In [5]:
import os 
import pandas as pd 
import sqlite3 
# Path to the downloaded files path = "/home/repl/.cache/kagglehub/datasets/lamyalbert/volve-production-data/versions/1"  # See what files are available 
print(os.listdir(path))

['Volve production data.xlsx']


In [7]:
excel_path = path + "/Volve production data.xlsx"
excel_file = pd.ExcelFile(excel_path)
print("Available sheets:", excel_file.sheet_names)

Available sheets: ['Daily Production Data', 'Monthly Production Data']


In [9]:
# Connect to a local SQLite database (it will create it automatically)
conn = sqlite3.connect('volve.db')

# Load the Excel sheets into pandas dataframes and save them as SQL tables
df_daily = pd.read_excel(excel_path, sheet_name='Daily Production Data')
df_monthly = pd.read_excel(excel_path, sheet_name='Monthly Production Data')

df_daily.to_sql('daily_production', conn, if_exists='replace', index=False)
df_monthly.to_sql('monthly_production', conn, if_exists='replace', index=False)

print("Tables successfully loaded into SQLite database: 'daily_production' and 'monthly_production'")

# Close connection
conn.close()

Tables successfully loaded into SQLite database: 'daily_production' and 'monthly_production'


In [ ]:
SELECT * 
FROM daily_production 
LIMIT 5;

Error: Please select a query source.

In [10]:
import sqlite3
import pandas as pd

# Connect to your database
conn = sqlite3.connect('volve.db')

# Write and run your SQL query
query = """
SELECT * 
FROM daily_production 
LIMIT 5;
"""

df_sample = pd.read_sql(query, conn)
conn.close()

# Display the preview
display(df_sample)

,DATEPRD,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,AVG_DP_TUBING,AVG_ANNULUS_PRESS,AVG_CHOKE_SIZE_P,AVG_CHOKE_UOM,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL,FLOW_KIND,WELL_TYPE
0,2014-04-07 00:00:00,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,0.00000,0.00000,0.00000,0.0,0.000000,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,None,production,WI
1,2014-04-08 00:00:00,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,NaN,NaN,0.0,1.003059,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,None,production,OP
2,2014-04-09 00:00:00,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,NaN,NaN,0.0,0.979008,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,None,production,OP
3,2014-04-10 00:00:00,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,NaN,NaN,0.0,0.545759,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,None,production,OP
4,2014-04-11 00:00:00,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,310.37614,96.87589,277.27826,0.0,1.215987,%,33.09788,10.47992,33.07195,0.0,0.0,0.0,None,production,OP


In [12]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('volve.db')

query = "SELECT DATEPRD, WELL_BORE_CODE, BORE_OIL_VOL, SUM(BORE_OIL_VOL) OVER (PARTITION BY WELL_BORE_CODE ORDER BY DATEPRD) AS cumulative_oil_production FROM daily_production WHERE BORE_OIL_VOL IS NOT NULL LIMIT 10;"

df_window = pd.read_sql(query, conn)
conn.close()

display(df_window)

,DATEPRD,WELL_BORE_CODE,BORE_OIL_VOL,cumulative_oil_production
0,2014-04-07 00:00:00,NO 15/9-F-1 C,0.0,0.0
1,2014-04-08 00:00:00,NO 15/9-F-1 C,0.0,0.0
2,2014-04-09 00:00:00,NO 15/9-F-1 C,0.0,0.0
3,2014-04-10 00:00:00,NO 15/9-F-1 C,0.0,0.0
4,2014-04-11 00:00:00,NO 15/9-F-1 C,0.0,0.0
5,2014-04-12 00:00:00,NO 15/9-F-1 C,0.0,0.0
6,2014-04-13 00:00:00,NO 15/9-F-1 C,0.0,0.0
7,2014-04-14 00:00:00,NO 15/9-F-1 C,0.0,0.0
8,2014-04-15 00:00:00,NO 15/9-F-1 C,0.0,0.0
9,2014-04-16 00:00:00,NO 15/9-F-1 C,0.0,0.0


In [13]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('volve.db')

query = "SELECT DATEPRD, WELL_BORE_CODE, BORE_OIL_VOL, SUM(BORE_OIL_VOL) OVER (PARTITION BY WELL_BORE_CODE ORDER BY DATEPRD) AS cumulative_oil_production FROM daily_production WHERE BORE_OIL_VOL > 0 LIMIT 10;"

df_window = pd.read_sql(query, conn)
conn.close()

display(df_window)

,DATEPRD,WELL_BORE_CODE,BORE_OIL_VOL,cumulative_oil_production
0,2014-04-22 00:00:00,NO 15/9-F-1 C,631.47,631.47
1,2014-04-23 00:00:00,NO 15/9-F-1 C,1166.46,1797.93
2,2014-04-24 00:00:00,NO 15/9-F-1 C,1549.81,3347.74
3,2014-04-25 00:00:00,NO 15/9-F-1 C,1248.70,4596.44
4,2014-04-26 00:00:00,NO 15/9-F-1 C,1345.78,5942.22
5,2014-04-27 00:00:00,NO 15/9-F-1 C,1349.56,7291.78
6,2014-04-28 00:00:00,NO 15/9-F-1 C,1345.61,8637.39
7,2014-04-29 00:00:00,NO 15/9-F-1 C,1279.46,9916.85
8,2014-04-30 00:00:00,NO 15/9-F-1 C,1225.62,11142.47
9,2014-05-01 00:00:00,NO 15/9-F-1 C,1212.90,12355.37


In [14]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('volve.db')

query = "SELECT DATEPRD, WELL_BORE_CODE, BORE_OIL_VOL, SUM(BORE_OIL_VOL) OVER (PARTITION BY WELL_BORE_CODE ORDER BY DATEPRD) AS cumulative_oil_production FROM daily_production WHERE BORE_OIL_VOL > 0 LIMIT 10;"

df_window = pd.read_sql(query, conn)
conn.close()

display(df_window)

,DATEPRD,WELL_BORE_CODE,BORE_OIL_VOL,cumulative_oil_production
0,2014-04-22 00:00:00,NO 15/9-F-1 C,631.47,631.47
1,2014-04-23 00:00:00,NO 15/9-F-1 C,1166.46,1797.93
2,2014-04-24 00:00:00,NO 15/9-F-1 C,1549.81,3347.74
3,2014-04-25 00:00:00,NO 15/9-F-1 C,1248.70,4596.44
4,2014-04-26 00:00:00,NO 15/9-F-1 C,1345.78,5942.22
5,2014-04-27 00:00:00,NO 15/9-F-1 C,1349.56,7291.78
6,2014-04-28 00:00:00,NO 15/9-F-1 C,1345.61,8637.39
7,2014-04-29 00:00:00,NO 15/9-F-1 C,1279.46,9916.85
8,2014-04-30 00:00:00,NO 15/9-F-1 C,1225.62,11142.47
9,2014-05-01 00:00:00,NO 15/9-F-1 C,1212.90,12355.37


In [15]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('volve.db')

query = """
SELECT 
    WELL_BORE_CODE,
    SUM(BORE_OIL_VOL) AS total_oil_produced,
    RANK() OVER (ORDER BY SUM(BORE_OIL_VOL) DESC) as production_rank
FROM daily_production
WHERE BORE_OIL_VOL IS NOT NULL
GROUP BY WELL_BORE_CODE;
"""

df_rank = pd.read_sql(query, conn)
conn.close()

display(df_rank)

,WELL_BORE_CODE,total_oil_produced,production_rank
0,NO 15/9-F-12 H,4579609.55,1
1,NO 15/9-F-14 H,3942233.39,2
2,NO 15/9-F-11 H,1147849.10,3
3,NO 15/9-F-1 C,177709.33,4
4,NO 15/9-F-15 D,148518.56,5
5,NO 15/9-F-5 AH,41160.68,6
